In [1]:
from vizdoom import *
import numpy as np
import random
import time

In [ ]:
# Demo with random actions
EPISODES = 5

game = DoomGame()
game.load_config('ViZDoom/scenarios/basic.cfg')
game.init()

actions = np.identity(3, dtype=np.uint8)

print('REWARDS')
print('--------------------')
for episode in range(EPISODES):
    game.new_episode()

    while not game.is_episode_finished():
        state = game.get_state()
        frame = state.screen_buffer
        info = state.game_variables
        reward = game.make_action(random.choice(actions))
        time.sleep(0.02)
    
    print(f'Episode {episode + 1} = {game.get_total_reward()}')
    time.sleep(2)

game.close()

REWARDS
--------------------
Episode 1 = -249.0
Episode 2 = -197.0
Episode 3 = -375.0
Episode 4 = -47.0
Episode 5 = 70.0


In [2]:
from gymnasium import Env
from gymnasium.spaces import Discrete, Box
import cv2

In [3]:
class VizDoomGym(Env):
    def __init__(self, render=False):
        super().__init__()
        self.game = DoomGame()
        self.game.load_config('ViZDoom/scenarios/basic.cfg')
        self.game.set_window_visible(render)
        self.game.init()

        self.observation_space = Box(low=0, high=255, shape=(100, 160, 1), dtype=np.uint8)
        self.action_space = Discrete(3)

    def step(self, action):
        actions = np.identity(3, dtype=np.uint8)
        reward = self.game.make_action(actions[action], 4)  # Skip 4 frames
        state = self.game.get_state()
        terminated = self.game.is_episode_finished()
        truncated = False

        if state:
            frame = state.screen_buffer
            frame = self.grayscale(frame)
            info = {'info': state.game_variables[0]}
        else:
            frame = np.zeros(self.observation_space.shape, dtype=np.uint8)
            info = {'info': 0}

        return frame, reward, terminated, truncated, info

    def render(self):
        pass

    def grayscale(self, observation):
        frame = cv2.cvtColor(np.moveaxis(observation, 0, -1), cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (160, 100), interpolation=cv2.INTER_CUBIC)
        frame = np.reshape(frame, (100, 160, 1))

        return frame

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.game.new_episode()
        state = self.game.get_state()

        if state is None:
            frame = np.zeros(self.observation_space.shape, dtype=np.uint8)
        else:
            frame = self.grayscale(state.screen_buffer)

        return frame, {}

    def close(self):
        self.game.close()


In [4]:
import os
from stable_baselines3.common.callbacks import BaseCallback

class TrainAndLoggingCallback(BaseCallback):
    def __init__(self, check_freq, save_path, verbose=1):
        super(TrainAndLoggingCallback, self).__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path

    def _init_callback(self):
        if self.save_path:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            model_path = os.path.join(self.save_path, f'best_model_{self.n_calls}')
            self.model.save(model_path)
        
        return True

In [5]:
CHECKPOINT_DIR = "./train/train_basic"
LOG_DIR = "./log/log_basic"

In [6]:
callback = TrainAndLoggingCallback(check_freq=10000, save_path=CHECKPOINT_DIR)

In [7]:
from stable_baselines3 import PPO
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [8]:
env = VizDoomGym()

In [ ]:
model = PPO('CnnPolicy', env, tensorboard_log=LOG_DIR, verbose=1, learning_rate=1e-4, n_steps=2048, device=device)

Using mps device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.


In [10]:
model.learn(total_timesteps=100000, callback=callback)

Logging to ./log/log_basic/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 43.8     |
|    ep_rew_mean     | -133     |
| time/              |          |
|    fps             | 67       |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 256      |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 49.7        |
|    ep_rew_mean          | -185        |
| time/                   |             |
|    fps                  | 48          |
|    iterations           | 2           |
|    time_elapsed         | 10          |
|    total_timesteps      | 512         |
| train/                  |             |
|    approx_kl            | 0.027735004 |
|    clip_fraction        | 0.277       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -0.000151  

KeyboardInterrupt: 